In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
from musicdb import PanDBIO
from gate import IOStore, IDStore
from match import MatchDBDataIO, AlbumReq, MatchString, MatchSeries, poolMatchSeries
from uuid import uuid4
from pandas import Series

In [ ]:
from ioutils import FileIO
io = FileIO()
baseNames = io.get("baseNames.p")
compNames = io.get("compNames.p")
print(len(baseNames),len(compNames))

In [ ]:
df = poolMatchSeries(baseNames.head(100), compNames, verbose=True, progress=True)

In [ ]:

# Process [Matching 2 x 251385] Ran For 2 Seconds    ==> Time Is 2022-04-18 13:41:12
# Process [Matching 20 x 251385] Ran For 6 Seconds    ==> Time Is 2022-04-18 13:41:39
# Process [Matching 100 x 251385] Ran For 18 Seconds    ==> Time Is 2022-04-18 13:42:17
Process [Matching 300 x 251385] Ran For 44 Seconds    ==> Time Is 2022-04-18 13:44:45

In [ ]:
from tqdm import tqdm

In [ ]:
help(tqdm)

In [ ]:
baseio = MatchDBDataIO(db="Spotify")
compareio = MatchDBDataIO(db="Discogs")
baseMedia = baseio.getData(albums=AlbumReq(min=3))
compareMedia = compareio.getData(albums=AlbumReq(min=3))
baseNames = baseMedia["Name"]
compNames = compareMedia["Name"]
io.save(idata=baseNames, ifile="baseNames.p")
io.save(idata=compNames, ifile="compNames.p")

In [ ]:
df

In [ ]:
baseNames = baseMedia["Name"]
compNames = compareMedia["Name"]
from ioutils import FileIO
io = FileIO()
io.save(idata=baseNames, ifile="baseNames.p")
io.save(idata=compNames, ifile="compNames.p")
from match import getLevenshtein
from numpy import vectorize
vLev = vectorize(getLevenshtein)
 
import pandas as pd
import swifter

In [ ]:
baseNames.apply
{baseid: for baseid,baseName in baseNames.iteritems()
 
 baseNames.swifter.apply()

df['outCol'] = df[['inCol1', 'inCol2', 'inCol3']].swifter.apply(my_func,
             positional_arg, keyword_arg=keyword_argval)

In [ ]:
from multiprocessing import Pool
from pandas import concat
from functools import partial

def poolMatchRunner(item, *args, **kwargs):
    return poolMatchSeries(base=item, compare=kwargs.get('comp'))
    
def poolMatchSeries(base, compare):
    assert isinstance(base,Series), "base must be a Series"
    assert isinstance(compare,Series), "compare must be a Series"
    retval = {baseid: compare.apply(getLevenshtein, x2=baseName) for baseid,baseName in base.iteritems()}
    return retval


In [ ]:

def poolMatch(item, *args, **kwargs):
def poolMatchRunner():
    return poolMatchSeries(kwargs[])
nCores = 2
bSplit = array_split(baseNames, nCores)
pool   = Pool(nCores)


df     = concat(pool.map(pfunc, bSplit))
pool.close()
pool.join()


In [ ]:

pfunc  = partial(poolMatchRunner, **kwargs)

In [ ]:
pfunc()

In [ ]:
##############################################################################################################################
# Pool Parse RawData Function
##############################################################################################################################
def poolParseIO(parseFunction, modVals=None, expr="< 0 Days", force=False, numProcs=3):
    argument_list = getModVals(modVals)
    print("poolIO(numProcs={0})".format(numProcs))
    print("  ==> ParseFunction: {0}".format(parseFunction.__name__))
    print("  ==> ModVals:       {0}".format(argument_list))
    kwargs = {"expr": expr, "force": force}
    pfunc  = partial(parseFunction, **kwargs)
    ts = Timestat("Running imap multiprocessing for {0} mod values ...".format(len(argument_list)))
    result_list = tqdmMap(func=pfunc, argument_list=argument_list, num_processes=numProcs)
    ts.stop()
    
    
def poolParse(item, *args, **kwargs):
    sleep(random())
    base = item
    matcher = kwargs['matcher']
    compare = kwargs['compare']
    del kwargs["matcher"]    
    #matcher(base, **kwargs)
    return matcher(base, compare)
    
def tqdmMap(func, argument_list, num_processes):
    pool = Pool(processes=num_processes)
    result_list_tqdm = []
    for result in tqdm(pool.imap(func=func, iterable=argument_list), total=len(argument_list)):
        result_list_tqdm.append(result)
    pool.close()
    pool.join()
    del pool
    return result_list_tqdm


     # Giving some arguments for kwargs
    kwargs = {"parser": parser, "expr": expr, "force": force}
    pfunc  = partial(func, **kwargs)

    ts = Timestat("Running imap multiprocessing for {0} mod values ...".format(len(argument_list)))
    result_list = tqdmMap(func=pfunc, argument_list=argument_list, num_processes=num_processes)
    ts.stop()
    
    

In [ ]:
baseMedia.head()

In [ ]:
compareMedia.head()

# LastFM <=> MusicBrainz

In [ ]:
mdbio = gate.getIO("LastFM")
mbrainz = gate.getIO("MusicBrainz")

In [ ]:
searchData = mdbio.data.getSearchArtistData()
searchData['id'] = searchData['url'].apply(mdbio.getdbid)
searchData['MB'] = searchData['mbid'].apply(mbrainz.getdbid)

In [ ]:
from pandas import Series
mbidmapData = searchData[searchData["MB"].notna() & searchData["id"].notna()].drop_duplicates(subset=["mbid","MB"])
mbidMap = Series(mbidmapData['id'].values, index=mbidmapData["MB"].values)

In [ ]:
mmeDF["LastFMNew"] = mmeDF["MusicBrainz"].map(mbidmap)

In [ ]:
mmeDF = mmeDF.drop(["LastFM"], axis=1)
mmeDF = mmeDF.rename(columns={"LastFMNew": "LastFM"})

# SetListFM <=> MusicBrainz

In [ ]:
def mergeSetListFM(row, slfmMap):
    retval = row["SetListFM"] if isinstance(row["SetListFM"],str) else slfmMap.get(row["MusicBrainz"], None)
    return retval

ios     = IOStore()
ids     = IDStore()
mio     = ios.get("SetListFM")
saData  = mio.data.getSearchArtistData()
saData['MB'] = saData['mbid'].apply(ids.getmbid)
slfmMap = saData["id"]
print("Found {0} SetListFM <=> MusicBrainz Map Entries".format(len(slfmMap)))

pdbio = PanDBIO()
mmeDF = pdbio.getData()
mmeDF["SetListFMNew"] = mmeDF[["MusicBrainz", "SetListFM"]].apply(mergeSetListFM, slfmMap=slfmMap, axis=1)
print("Found {0} Previous SetListFM Values".format(mmeDF["SetListFM"].count()))
print("Found {0} New SetListFM Values".format(mmeDF["SetListFMNew"].count()))

In [ ]:
mmeDF = mmeDF.drop(["SetListFM"], axis=1)
mmeDF = mmeDF.rename(columns={"SetListFMNew": "SetListFM"})

In [ ]:
pdbio.saveData(mmeDF)

# Beatport From MusicBrainz

In [ ]:
from gate import MusicDBGate
gate  = MusicDBGate()
mdbio = gate.getIO("MusicBrainz")
bpio  = gate.getIO("Beatport")
bpMap = {}
mbMap = {}
for modVal in range(100):
    modValData = mdbio.data.getModValData(modVal)
    for mbid,mbidData in modValData.iteritems():
        if mbidData.profile.external is None:
            continue
        for item in mbidData.profile.external.get("Beatport", []):
            ref  = item.href if item is not None else None
            bpid = bpio.getdbid(ref) if isinstance(ref,str) else None
            if bpMap.get(mbid) is None:
                bpMap[mbid] = {}
            bpMap[mbid][(bpid,ref)] = mbidData.artist.name
    if modVal % 5 == 0:
        print(modVal,'\t',len(bpMap),'\t',len(mbMap))
bpMap = Series(bpMap)

In [ ]:
from pandas import Series,DataFrame
s = Series(bpMap)
beatportMap = {}
for k,v in bpMap[bpMap.map(len) == 1].iteritems():
    for k2,v2 in v.items():
        beatportMap[k] = k2[0]

for k,v in bpMap[bpMap.map(len) > 1].iteritems():
    for k2,v2 in v.items():
        print("beatportMap['{0}'] = {1: <10}  ## {2}  /  {3}".format(k,"'{0}'".format(k2[0]),k2[1],v2))
    print("")

In [ ]:
s = Series(beatportMap)
s.name = "Beatport"
df = DataFrame(s).join(mdbio.data.getSummaryNameData())

In [ ]:
externalsToGet = df[~df.index.isin(pdbio.getNotNaDBIDs("MusicBrainz")["MusicBrainz"])]
from ioutils import FileIO
io = FileIO()
io.save(idata=externalsToGet, ifile="beatportMap.p")

In [ ]:
for mbid,row in df[~df.index.isin(pdbio.getNotNaDBIDs("MusicBrainz")["MusicBrainz"])].iterrows():
    artistName = row["Name"]
    bpid = row["Beatport"]
    pdbio.newArtist(name=artistName, Beatport=bpid, MusicBrainz=mbid)

In [ ]:
toadd = externalsToGet.reset_index()
toadd = toadd.rename(columns={"index": "MusicBrainz", "Name": "ArtistName"})

In [ ]:
for i in range(toadd.shape[0]):

In [ ]:
from uuid import uuid4
idx = []
for i in range(toadd.shape[0]):
    idx.append(str(uuid4()))
toadd.index = idx

In [ ]:
from pandas import concat
pdbio.saveData(concat([mmeDF,toadd]))

In [ ]:
mmeDF["Beatport"] = mmeDF["MusicBrainz"].map(beatportMap)

In [ ]:
pdbio.saveData(mmeDF)

In [ ]:
mdioData

# Other

In [ ]:

        try:
            summaryCountsData = mdbio.data.getSummaryCountsData()
        except:
            print("Could not get SummaryCounts Data")
            summaryCountsData = None
            
        try:
            summaryCountsData = summaryCountsData[mediaTypes].rename(columns={col: "Num{0}".format(col) for col in summaryCountsData.columns})
            summaryCountsData["NumMedia"] = summaryCountsData.sum(axis=1)
        except:

In [ ]:
ios = IOStore()
dbAlbums = {}
for db,mdbio in ios.get().items():
    countsData = mdbio.data.getSummaryCountsData()
    break

In [ ]:
countsData

In [ ]:

    dbAlbums[db] = numAlbums if isinstance(numAlbums,Series) else Series(dtype='object')
dfAlbums = DataFrame({db: to_numeric(dbIDMatches.apply(dbAlbums[db].get), errors='coerce') for db,dbIDMatches in pdbio.mmeDF.items() if db in dbAlbums})
dfRank   = DataFrame({db: dbIDs[dbIDs.notna()].rank(pct=True) for db,dbIDs in dfAlbums.items()}).fillna(0.0).mean(axis=1).rank(pct=True)
dfRank.name = "Rank"


In [ ]:

####################################################################################################################    
## Append Metadata To DataFrame
####################################################################################################################
def addMetrics(self):
    ts = Timestat("Adding Metrics To PanDB")
    ios = IOStore()

    ######################################################################
    # Get PanDB
    ######################################################################
    pdbio.setData()

    ######################################################################
    # Calculations
    ######################################################################
    ######################################################################
    # Join Rank
    ######################################################################
    if "Rank" in pdbio.mmeDF.columns:
        pdbio.mmeDF.drop(["Rank"], axis=1, inplace=True)
    pdbio.mmeDF         = pdbio.mmeDF.join(dfRank)
    pdbio.mmeDF["Rank"] = pdbio.mmeDF["Rank"].fillna(0.0).rank(method='max', ascending=False).apply(int)

    ######################################################################
    # Join Albums
    ######################################################################        
    if "Albums" in pdbio.mmeDF.columns:
        pdbio.mmeDF.drop(["Albums"], axis=1, inplace=True)
    dfAllAlbums = dfAlbums.sum(axis=1).fillna(0).astype(int)
    dfAllAlbums.name = "Albums"
    pdbio.mmeDF           = pdbio.mmeDF.join(dfAllAlbums)
    pdbio.mmeDF["Albums"] = to_numeric(pdbio.mmeDF["Albums"], errors='coerce').fillna(0)

    ######################################################################
    # Join Counts
    ######################################################################        
    if "Counts" in pdbio.mmeDF.columns:
        pdbio.mmeDF.drop(["Counts"], axis=1, inplace=True)
    pdbio.mmeDF["Counts"] = pdbio.mmeDF.drop(["Rank", "Albums", "ArtistName"], axis=1).count(axis=1)

    ts.stop()


In [ ]:
pdbio = PanDBIO()
mmeDF = pdbio.getData()
pdbNames = mmeDF["ArtistName"]

In [ ]:
from match import MatchCounts, MatchDBCounts
db = "Discogs"
unMatched = MatchDBCounts(db).getUnmatched()
dbNames   = unMatched.head(5)["ArtistName"]

In [ ]:
mmeDF[mmeDF["ArtistName"].str.contains("m McLean")]

In [ ]:


pdbio.saveData()

In [ ]:
for artistID,name in dbNames.iteritems():
    print("")
    print("#="*150)
    match = MatchString(base=name, compare=pdbNames).get(0.9)
    print("pdbio.newArtist('{0}', {1}='{2}')".format(name, db, artistID))
    print("pdbio.setdbid('', '{0}', '{1}')".format(db, artistID))
    if len(match) > 0:
        pdbData = pdbio.getRows(match.keys()).drop(['SetListFM', 'AlbumOfTheYear', 'Deezer', 'LastFM'], axis=1).to_string(index=True)
        print(pdbData)
    print("\n\n")

In [ ]:
https://www.setlist.fm/setlists/orchester-der-wiener-volksoper-7bde12d0.html

In [ ]:

ids.getmbid('https://musicbrainz.org/artist/d30820d5-d1c5-4c16-a110-8c59ad6e352c')

In [ ]:
#======================================================================================================================================================
pdbio.newArtist('Wiener Volksopernorchester', Discogs='833376', SetListFM='7bde12d0', Spotify='1gCaCu90v4PYeYxWq0joIW', RateYourMusic='308850', MusicBrainz='10145681830259485259250037629476827214')
pdbio.newArtist('Radio-Symphonie-Orchester Berlin', Discogs='688716', MusicBrainz='189002019997078465049180482273862734719', AllMusic='0002221657', Spotify='49TgMBH68KIFiOmLMoUOWY')
pdbio.newArtist('Staatskapelle Berlin', Discogs='833446', MusicBrainz='133018627337105403763302852079106379613', AllMusic="0002059537", Spotify="7vEPPI71V8dEHtEhPMAxWT", RateYourMusic="336903")
#======================================================================================================================================================
pdbio.newArtist("The King's College Choir Of Cambridge", Discogs='700443', MusicBrainz='273338754289470679440863230907769944743', AllMusic='0000608992', Spotify='0f3PsS9IQ6whvNMFFKnpjl', RateYourMusic='44555')
pdbio.newArtist('Hugo Winterhalter Orchestra', Discogs='395324', MusicBrainz='168341810660794848268304798831052693483', Spotify='5WVQxyLSWuV7XpjDlgNY53', RateYourMusic='55184')
pdbio.saveData()

In [ ]:
pdbio.saveData()

In [ ]:
unMatched[unMatched["ArtistName"].str.contains("Group")].head(10)

In [ ]:
for artistID,row in mdbc.getUnmatched().head(40).tail(20).iterrows():
    print("pdbio.newArtist('{0}', Genius='{1}', Spotify='')".format(row["ArtistName"], artistID))

In [ ]:
match

In [ ]:
for 

In [ ]:
pdbio.newArtist('Adventure Archives', Genius='2429680', Spotify='')
pdbio.newArtist('Home Runs of the Day', Genius='59781', Spotify='')
pdbio.newArtist('Rudy van Gelder', Genius='641939', Spotify='')
pdbio.newArtist('Chris Lord-Alge', Genius='108681', Spotify='')
pdbio.newArtist('Phil Elverum', Genius='666880', Spotify='')
pdbio.newArtist('La Vendicion', Genius='1194294', Spotify='')
pdbio.newArtist('Todd Tobias', Genius='28780', Spotify='')
pdbio.newArtist('EMI Music Publishing', Genius='648041', Spotify='')
pdbio.newArtist('Danja', Genius='10461', Spotify='')
pdbio.newArtist('Peter Asher', Genius='40509', Spotify='')
pdbio.newArtist('Raphael (Spanish singer)', Genius='1259329', Spotify='')
pdbio.newArtist('Creed Taylor', Genius='577179', Spotify='')
pdbio.newArtist('Ori Shochat - אורי שוחט', Genius='573979', Spotify='')
pdbio.newArtist('Dave Kutch', Genius='641341', Spotify='')
pdbio.newArtist('Tony Visconti', Genius='37268', Spotify='')
pdbio.newArtist('Honiro Label', Genius='1158245', Spotify='')
pdbio.newArtist('Chris Thomas', Genius='45700', Spotify='')
pdbio.newArtist('Benmont Tench', Genius='497398', Spotify='')
pdbio.newArtist('Tricky Stewart', Genius='27603', Spotify='')
pdbio.saveData()

In [ ]:
pdbio.newArtist('Rick Bonadio', Genius='192504', Spotify='2CZ8dMcFFZ1UYj52mUSaE6', Discogs='252040', RateYourMusic='1016711')
pdbio.newArtist('Nigel Godrich', Genius='38556', Spotify='0g7gHEXKEHU4snTwOZSxNO', AllMusic='0000869688', RateYourMusic='427029', Discogs='169094')
pdbio.newArtist('DooMee', Genius='1105756', Spotify='0Rb1j4P056IYg6ncXqslRr', RateYourMusic='1521674')
pdbio.newArtist('Mike Green', Genius='29122', Spotify='3kK0N0qZ5yuwWsqtaOfcQm', AllMusic='0000221368', RateYourMusic='1228102')
pdbio.newArtist('Prodigy', Genius='122', Spotify='1GwxXgEc6oxCKQ5wykWXFs', MusicBrainz='235890702021636578201118651139050295121', Discogs='134136', AllMusic="0000855953", RateYourMusic="27449")
pdbio.newArtist('Wheezy', Genius='339721', Spotify='4Ufo9whpMn1BwjnB3MJSYL', AllMusic='0002309613', RateYourMusic='1137548')
pdbio.newArtist('Benihana Boy', Genius='2361969', Spotify='0rygaaiuUiGN9W1o0mH6HV', AllMusic='0003796658', Discogs='7433745')
pdbio.newArtist('Mike McCready', Genius='372621', Spotify='7njqqUBXHc5fpyXmUlfOUL', AllMusic='0000491447', MusicBrainz='44313753520174712438202403431622254380', RateYourMusic="748912", Discogs='275980')
pdbio.saveData()

In [ ]:
ids = IDStore()
ids.getmbid("https://musicbrainz.org/artist/16cb4a74-225a-4262-a885-628083f12460")
#pdbio.getmbid()

In [ ]:
pdbio.saveData()

In [ ]:
baseDB    = "RateYourMusic"  # 2
baseDB    = "Genius"  # 155
baseReq   = {baseDB: AlbumReq(top=2000)}

compareDBs  = ["RateYourMusic", "LastFM", "Spotify", "Genius", "Discogs", "MusicBrainz", "Deezer", "MetalArchives"]
compareDBs  = ["RateYourMusic", "Spotify", "Genius", "Discogs", "MusicBrainz"]
compareDBs  = ["RateYourMusic", "Spotify", "Discogs", "MusicBrainz"]
compareReqs = {compareDB: AlbumReq(min=3) for compareDB in compareDBs if compareDB not in [baseDB]}
compareDBs  = list(compareReqs.keys())

albumReqs = {**baseReq, **compareReqs}
mediaTypes = ["Album", "SingleEP"]
mediaTypes = ["{0}Media".format(mediaType) for mediaType in mediaTypes]
reqs       = {"Media": mediaTypes, "Albums": albumReqs, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.8, "Medium": 3, "Tight": 1}}
print("Run Params:")
print("  ==> DBs:   [{0}] x {1}]".format(baseDB,list(compareReqs.keys())))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(reqs["Match"]))

In [ ]:
pdbio.g

In [ ]:
mc = MatchCounts()

In [ ]:
from pandas import Series


In [ ]:
searchData["Matched"].sum()

In [ ]:
pio = PanDBIO() 
pio.getTraxsourceIO()

In [ ]:
""" PanDB MusicDBID Gate """

__all__ = ["MusicDBGate"]

from importlib import import_module
from master import MasterDBs

In [ ]:
pdbid = PanDBID()
pdbid.getmbid("dkfjd")
#pdbid.get("MusicBrainz", )

In [ ]:
pdbid.getmbid('https://musicbrainz.org/artist/a8c25ccf-2e1a-4b7b-a2ad-230f40c14a74')

In [ ]:
mmeDF[mmeDF["RateYourMusic"] == '237673']

In [ ]:
mdbc.searchData.loc['237673']

In [ ]:
DataFrame(mdbc.matchedIDs).value_counts()

In [ ]:
for col,colData in mmeDF.iteritems():
    if mdbs.isValid(col):
        values  = colData[colData.notna()].to_list()
        mdbdata = MusicDBData(path=MusicDBDir(mdbpd.getMusicDBPermPath()), fname=col)
        mdbdata.save(data=values)

In [ ]:
entries = {}
artistName = None
for key,value in mmeDF.tail(448)[0].iteritems():
    if key == "ArtistName":
        artistName = value
        entries[artistName] = {}
    else:
        entries[artistName][key] = value

In [ ]:
df2 = DataFrame(entries).T
df2 = df2.reset_index()
df2 = df2.rename(columns={"index": "ArtistName"})
df2.index = [str(uuid4()) for i in df2.index]
mmeDF = concat([mmeDF,df2], axis=0)

In [ ]:
ios = IOStore()
mdbio = ios.get("Genius")
mdbio.search.make(verbose=True)

In [ ]:
from apiutils import WebIO
from ioutils import HTMLIO
wio = WebIO()

In [ ]:
url='https://www.setlist.fm/artist/browse/a/1.html'
retval = wio.get(url)

In [ ]:
bsdata = HTMLIO().get(retval.getData())

In [ ]:
bsdata

# Fix Match

In [ ]:
""" Main Matching Class """

__all__ = ["MatchDB"]

from timeutils import Timestat
from listUtils import getFlatList
from match.matchlev import getLevenshtein
from match.dataio import MatchDBDataIO
from match.albumreq import AlbumReq
from match.utils import write
from pandas import DataFrame, Series, concat
import dask.dataframe as dd
from dask.diagnostics import ProgressBar

class MatchDB:
    def __init__(self, baseDB: str, compareDBs: list, reqs: dict, **kwargs):
        self.verbose = kwargs.get('verbose', True)
        if self.verbose:
            print("#"*175)
            print("{0}  {1}  {2}".format("#"*70, "MatchDB()", "#"*(175-70-4-len("MatchDB()"))))
            print("#"*175)
            
        mask = reqs.get("Mask")
        assert isinstance(mask,(bool,str)), "Mask Req is not set."
        
        mediaTypes = reqs.get("Media")
        assert isinstance(mediaTypes,list) or mediaTypes is None, "Media Req is not set."
        self.mediaTypes = mediaTypes
        
        albumReqs = reqs.get("Albums")
        assert isinstance(albumReqs,dict), "Albums Req is not set."
        self.albumReqs = albumReqs
        
        self.nPart = reqs.get("NPart", 3)
        assert isinstance(self.nPart,int), "NPart Req is not set."
        
        self.matchReqs = reqs.get("Match")
        assert isinstance(self.matchReqs,dict), "Match Req is not set."
        assert isinstance(self.matchReqs.get('Artist'),float), "Artist match req is not set"
        assert isinstance(self.matchReqs.get('Medium'),int), "Artist match req is not set"
        assert isinstance(self.matchReqs.get('Tight'),int), "Artist match req is not set"
        
        self.baseIO = MatchDBDataIO(db=baseDB, mediaTypes=mediaTypes, mask=mask, verbose=False)
        assert isinstance(self.albumReqs.get(self.baseIO.db),AlbumReq), "Reqs does not have BaseDB [{0}]".format(baseIO.db)
        
        self.compareIOs = {}
        for compareDB in compareDBs:
            compareIO = MatchDBDataIO(db=compareDB, mediaTypes=mediaTypes, mask=mask, verbose=False)
            assert isinstance(self.albumReqs.get(compareIO.db),AlbumReq), "Reqs does not have CompareDB [{0}]".format(compareIO.db)
            self.compareIOs[compareDB] = compareIO
        
        self.diagnostics = {}
        
        pbar = ProgressBar()
        pbar.register()
        

    def match(self, **kwargs):
        verbose = kwargs.get('verbose', self.verbose)                
        tsMatch = Timestat("Matching [{0}] Against {1}".format(self.baseIO.db, list(self.compareIOs.keys())), ind=0)
        
        print("")
        print("-"*150)
        print("{0} {1} {2}".format("-"*70, self.baseIO.db, "-"*(150-70-2-len(self.baseIO.db))))
        baseIO = self.baseIO
        baseMediaMatchData = baseIO.getData(albums=self.albumReqs[baseIO.db])
        
        results = {}
        for compareDB,compareIO in self.compareIOs.items():
            print("")
            print("-"*150)
            print("{0} {1} {2}".format("-"*70, compareDB, "-"*(150-70-2-len(compareDB))))
            compareMediaMatchData = compareIO.getData(albums=self.albumReqs[compareIO.db])
            self.diagnostics[compareDB] = {}
        
            ########################################################################################################################################
            ## 1) Match Artist Names
            ########################################################################################################################################
            if verbose: ts = Timestat("String Matching {0} [{1}] x {2} [{3}] Artist Names".format(baseMediaMatchData.shape[0], baseIO.db, compareMediaMatchData.shape[0], compareIO.db), ind=2)
            daskDF = dd.from_pandas(baseMediaMatchData["Name"], npartitions=self.nPart)
            artistMatchResults = daskDF.map_partitions(lambda df: df.apply(lambda artistName: compareMediaMatchData["Name"].apply(getLevenshtein, x2=artistName))).compute(scheduler='processes')
            artistNameMatches  = self.selectArtistsForMediaMatch(artistMatchResults, self.matchReqs['Artist'])
            mediaData          = self.prepareMediaData(artistNameMatches, baseMediaMatchData, compareMediaMatchData)
            del artistMatchResults
            del artistNameMatches
            if verbose: ts.stop()
            
            
            ########################################################################################################################################
            ## 2) Match Artist Albums Names
            ########################################################################################################################################
            if verbose: ts = Timestat("String Matching {0} [{1}] Album Names".format(mediaData.shape[0], baseIO.db), ind=2)
            albumMatchResults = self.matchMediaData(mediaData)
            matchResults      = self.selectMatches(albumMatchResults)
            del albumMatchResults
            del mediaData
            if verbose: ts.stop()
                

            ########################################################################################################################################
            ## 3) Cross Match Data
            ########################################################################################################################################
            compareCrossMatchIDs = getFlatList(matchResults["Single"].apply(lambda x: x.index).values)
            if len(compareCrossMatchIDs) == 0:
                write(2, "Did not find any potential matches. Stopping process")
                tsMatch.update()
                continue
            baseMediaCrossMatchData    = compareIO.getData(ids=compareCrossMatchIDs)
            compareMediaCrossMatchData = baseIO.getData(albums=AlbumReq(min=2))
            
        
            ########################################################################################################################################
            ## 4) Cross Match Artist Names
            ########################################################################################################################################            
            if verbose: ts = Timestat("String Matching {0} [{1}] x {2} [{3}] Artist Names".format(baseMediaCrossMatchData.shape[0], compareIO.db, compareMediaCrossMatchData.shape[0], baseIO.db), ind=2)
            daskDF = dd.from_pandas(baseMediaCrossMatchData["Name"], npartitions=self.nPart)
            artistMatchResults = daskDF.map_partitions(lambda df: df.apply(lambda artistName: compareMediaCrossMatchData["Name"].apply(getLevenshtein, x2=artistName))).compute(scheduler='processes')
            artistNameMatches  = self.selectArtistsForMediaMatch(artistMatchResults, self.matchReqs["Artist"])
            mediaData          = self.prepareMediaData(artistNameMatches, baseMediaCrossMatchData, compareMediaCrossMatchData)
            del artistMatchResults
            del artistNameMatches
            if verbose: ts.stop()
            
            
            ########################################################################################################################################
            ## 5) Cross Match Artist Albums Names
            ########################################################################################################################################
            if verbose: ts = Timestat("String Matching {0} [{1}] Album Names".format(mediaData.shape[0], compareIO.db), ind=2)
            albumCrossMatchResults = self.matchMediaData(mediaData)
            crossMatchResults      = self.selectMatches(albumCrossMatchResults)
            del mediaData
            del albumCrossMatchResults
            if verbose: ts.stop()
            
            
            ########################################################################################################################################
            ## 6) Get Final Matches
            ########################################################################################################################################
            baseCrossMatchIDs = crossMatchResults["Single"].apply(lambda matchResult: list(matchResult.index)[0])
            crossMatchDF = DataFrame(matchResults["Single"].apply(lambda matchResult: list(matchResult.index)[0]), columns=["CompareID"])
            crossMatchDF["BaseIDMap"] = crossMatchDF["CompareID"].map(baseCrossMatchIDs)
            correctMatches = crossMatchDF[crossMatchDF.index == crossMatchDF["BaseIDMap"]]
            results[compareDB] = correctMatches
            del baseCrossMatchIDs
            del crossMatchDF
            
            tsMatch.update()
        
        self.results = results
        tsMatch.stop()
            

    def matchMediaData(self, artistNameMatches: DataFrame) -> 'Series':
        mediaResults = {}
        rankValues   = {"Loose": 0.7, "Medium": 0.8, "Tight": 0.9, "Exact": 0.95}
        for n,(baseID,nameMatchData) in enumerate(artistNameMatches.iterrows()):
            mediaResults[baseID] = {}

            baseNameMedia    = nameMatchData["BaseMedia"]
            baseMedia        = Series(baseNameMedia["Media"])
            compareNameMedia = Series(nameMatchData["CompareMedia"])

            ###############################################################################################
            ## Master Media String Comparisons
            ###############################################################################################
            compareResults = compareNameMedia.apply(lambda cNameMedia: Series(cNameMedia["Media"]).apply(lambda cMediaValue: baseMedia.apply(getLevenshtein, x2=cMediaValue)))
            for compareID,compareIDResult in compareResults.iteritems():
                bestBaseMatch    = Series(compareIDResult.max(axis=1).values, index=compareNameMedia[compareID]["Media"])
                bestCompareMatch = Series(compareIDResult.max(axis=0).values, index=baseNameMedia["Media"])

                baseRankResult = {rank: bestBaseMatch[bestBaseMatch >= value].count() for rank,value in rankValues.items()}
                compareRankResult = {rank: bestCompareMatch[bestCompareMatch >= value].count() for rank,value in rankValues.items()}

                rankData = concat([Series(baseRankResult, name="Base"), Series(compareRankResult, name="Compare")], axis=1)

                mediaResults[baseID][compareID] = {"Names": {"Base": baseNameMedia["Name"], "Compare": compareNameMedia[compareID]["Name"]},
                                                   "Rank": rankData, 
                                                   "Raw": {"BestBaseMatch": bestBaseMatch, "BestCompareMatch": bestCompareMatch}}

        retval = Series(mediaResults)
        return retval



    ################################################################################################################################################
    ## Utility Functions
    ################################################################################################################################################
    def selectArtistsForMediaMatch(self, artistMatchResults: DataFrame, artistNameCutoff: float) -> 'Series':        
        nearArtistNameMatches = artistMatchResults.apply(lambda values: values[values >= artistNameCutoff].to_dict(), axis=1)
        if self.verbose: write(4, "Found {0} Name Results", nearArtistNameMatches.shape[0])
        artistNameMatches = nearArtistNameMatches[nearArtistNameMatches.apply(len) > 0]
        if self.verbose: write(4, "Found {0} Artists With One Or More Matches", artistNameMatches.shape[0])
        if self.verbose: write(4, "Found {0} Possible Matches", artistNameMatches.apply(len).sum())
        return artistNameMatches

    
    def prepareMediaData(self, artistNameMatches: Series, baseMediaMatchData: DataFrame, compareMediaMatchData: DataFrame) -> 'DataFrame':
        def getMatchMediaData(row: Series) -> 'dict':
            rowMediaValues = {mediaType: row.get("{0}Media".format(mediaType),[]) for mediaType in self.mediaTypes}
            rowMediaValues = getFlatList([mediaTypeData for mediaTypeData in rowMediaValues.values() if isinstance(mediaTypeData,list)])
            return {"Name": row["Name"], "Media": list(set(rowMediaValues))}
        
        try:
            baseMediaValues = baseMediaMatchData.loc[artistNameMatches.index]
        except:
            print("artistNameMatches:")
            print(artistNameMatches.head())
            print(artistNameMatches.index)
            print("baseMediaMatchData:")
            print(baseMediaMatchData.head())
            raise ValueError("Error subselecting artists for media gathering")

            
        try:
            baseMediaValues = baseMediaValues.apply(lambda row: getMatchMediaData(row), axis=1)
        except:
            print("self.mediaTypes:")
            print(self.mediaTypes)
            print("baseMediaValues:")
            print(baseMediaValues.head())
            raise ValueError("Error calling baseMediaValues = baseMediaValues.apply(lambda row: getMatchMediaData(row), axis=1)")
        #print("")
        #print(type(baseMediaValues))
        #print(baseMediaValues.head())
        compareMediaValues = artistNameMatches.apply(lambda compareIDMatches: {compareID: getMatchMediaData(compareMediaMatchData.loc[compareID]) for compareID in compareIDMatches.keys()})
        #print("")
        #print(type(compareMediaValues))
        #print(compareMediaValues.head())

        try:
            baseMediaValues.name = "BaseMedia"
            compareMediaValues.name = "CompareMedia"
            retval = DataFrame(baseMediaValues).join(compareMediaValues)
            #retval = concat([baseMediaValues,compareMediaValues], axis=1)
            #retval.columns = ["BaseMedia", "CompareMedia"]    
        except:
            self.diagnostics["ConcatError"] = {"Base": baseMediaValues, "Compare": compareMediaValues}
            raise ValueError("Something went really wrong when running retval = concat([baseMediaValues,compareMediaValues], axis=1) in prepareMediaData(). See diagnotic for data")
        return retval
    
    def selectMatches(self, albumMatchResults: Series) -> 'dict':
        rankResults = albumMatchResults.apply(lambda x: DataFrame({compareID: compareIDResult["Rank"].min(axis=1) for compareID,compareIDResult in x.items()}).T)
        nearResults = rankResults.apply(lambda df: df[((df["Medium"] <  self.matchReqs["Medium"]) & (df["Tight"] >= self.matchReqs["Tight"])) |  
                                                      ((df["Medium"] >=  self.matchReqs["Medium"]) & (df["Tight"] < self.matchReqs["Tight"]))])
        goodResults = rankResults.apply(lambda df: df[(df["Medium"] >= self.matchReqs["Medium"]) & (df["Tight"] >= self.matchReqs["Tight"])])

        multipleMatches = goodResults[goodResults.apply(lambda df: df.shape[0]) > 1]
        singleMatches   = goodResults[goodResults.apply(lambda df: df.shape[0]) == 1]
        nearMatches     = nearResults[nearResults.apply(lambda df: df.shape[0]) >= 1]
        write(4, "Found [{0} / {1} / {2}] Multi/Good/Near Matches", (multipleMatches.shape[0], singleMatches.shape[0], nearMatches.shape[0]))

        retval = {"Multiple": multipleMatches, "Single": singleMatches, "Near": nearMatches}
        return retval